# 05.6 — The two-stage lifecycle

The Optimal training isn't one loop — it's **two stages**. Stage 1 trains an autoencoder *unsupervised* (reconstruct the input, no labels) so the encoder learns good representations from all the data. Stage 2 then trains the *full composite* supervised, starting from those pre-trained encoder weights. The bridge between them — copying Stage 1's weights into Stage 2 — is the interesting mechanic, and a clean showcase of `state_dict` surgery ([02.6](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)).

This notebook covers:

* Why pre-train unsupervised before supervised fine-tuning.
* Stage 1 (autoencoder) vs Stage 2 (composite) — what differs.
* The weight handoff: `copy_autoencoder_weights`.
* Per-stage checkpointing and resume.

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb) (state_dict), [05.5 checkpoints](05.5_checkpoint_resume_state_machine.ipynb).

## Section 1 — What MATLAB does

`cgg_runAutoEncoder` sequences two training phases:

```matlab
% Stage 1: train the autoencoder to RECONSTRUCT (no labels needed)
autoencoder = trainAutoEncoder(data, NumEpochsAutoEncoder);   % reconstruction loss only

% Handoff: seed the full network's encoder with the pre-trained weights
net = buildComposite(autoencoder, classifier);

% Stage 2: train the WHOLE network SUPERVISED (reconstruction + classification)
net = trainNetwork(data, labels, net, NumEpochsFull);
```

`cfg.NumEpochsAutoEncoder = 0` skips Stage 1 (train supervised from scratch); a positive value runs the two-stage lifecycle. The port mirrors this: `fit_two_stage` runs Stage 1, hands off the weights, then Stage 2.

## Section 2 — The Python concepts you need

### 2.1 — Why pre-train unsupervised

Labels are scarce and expensive; raw trials are plentiful. Unsupervised pre-training lets the encoder learn structure from *all* the data (by reconstructing it) before the smaller supervised signal fine-tunes it for the decoding task. The encoder arrives at Stage 2 already knowing "what neural data looks like" — so supervised training starts from a good representation instead of random weights. It's transfer learning within one pipeline.

### 2.2 — Stage 1 vs Stage 2

| | Stage 1 (autoencoder) | Stage 2 (composite) |
|---|---|---|
| goal | reconstruct the input | classify (+ reconstruct) |
| labels | **not used** | used |
| loss | reconstruction (+ KL) | reconstruction + KL + classification + confidence |
| modules | encoder, bottleneck, sampling, decoder | all of Stage 1 **+ classifier** |
| output | a trained autoencoder | the full decoder |

Stage 2's model is a *superset* of Stage 1's — same encoder/bottleneck/decoder, plus a classifier head. That structural overlap is exactly what makes the weight handoff possible.

### 2.3 — The handoff, live

The bridge copies Stage 1's four autoencoder submodules into Stage 2's composite, then Stage 2 fine-tunes them alongside a fresh classifier:

In [ ]:
import torch
from neural_data_decoding.models.composite import (
    build_variational_autoencoder, build_variational_composite, copy_autoencoder_weights,
)

cfg = {"in_features": 8, "samples_per_window": 1, "num_areas": 1,
       "hidden_sizes": [16, 4], "num_classes_per_dim": [3],
       "classifier_hidden_size": [8], "transform": "GRU", "dropout": 0.0,
       "want_normalization": False, "activation": "", "loss_type_decoder": "MSE",
       "classifier_dropout": 0.5, "confidence_type": [], "stitching_and_fusion_layer": ""}

stage1 = build_variational_autoencoder(cfg)     # encoder + bottleneck + sampling + decoder
stage2 = build_variational_composite(cfg)        # ...all of that PLUS a classifier

# Pretend Stage 1 trained: perturb its encoder weights so they're distinctive.
with torch.no_grad():
    for p in stage1.encoder.parameters():
        p.add_(1.0)

# BEFORE handoff: the two encoders differ (Stage 2's are freshly initialized).
enc1 = next(stage1.encoder.parameters())
enc2 = next(stage2.encoder.parameters())
print("before handoff — encoders equal?", torch.equal(enc1, enc2))

In [ ]:
# The handoff: copy Stage 1's autoencoder weights into Stage 2's composite.
copy_autoencoder_weights(stage1, stage2)

enc2_after = next(stage2.encoder.parameters())
print("after handoff — encoders equal? ", torch.equal(enc1, enc2_after))
print("Stage 2's encoder now STARTS from Stage 1's learned weights.")

One call transplants the pre-trained encoder, bottleneck, sampling, and decoder from the autoencoder into the composite. Stage 2 then trains the *whole* composite — fine-tuning those transplanted weights while learning the classifier head from scratch. The classifier is the only part that starts random; everything upstream is warm.

### 2.4 — Why `state_dict` copy, not a shared reference

The handoff *copies* weights (via `load_state_dict`), it doesn't share tensors — so after the handoff, Stage 2's encoder is independent. Fine-tuning it in Stage 2 doesn't retroactively change the saved Stage 1 autoencoder (which the Optimal-checkpoint deliverable might still reference). Confirm the independence:

In [ ]:
# After handoff, modifying Stage 2's encoder must NOT touch Stage 1's:
with torch.no_grad():
    next(stage2.encoder.parameters()).add_(100.0)
print("Stage 1 encoder unchanged by Stage 2 edit?",
      not torch.equal(next(stage1.encoder.parameters()), next(stage2.encoder.parameters())))

## Section 3 — The `neural_data_decoding` implementation

`copy_autoencoder_weights` — the four-submodule transplant:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/composite.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip().startswith("dst.encoder.load_state_dict"))
for k in range(i, min(i + 6, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Four `dst.X.load_state_dict(src.X.state_dict())` calls — encoder, bottleneck, sampling, decoder — the §2.2 shared submodules. The classifier is deliberately absent (it starts fresh).
* `load_state_dict` requires matching keys/shapes, so both models MUST be built from the same cfg — the docstring makes that the caller's responsibility, and a mismatch raises `RuntimeError` ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)).
* Optional submodules (pre-encoder, post-decoder, the CC.6 augmentation head) are handed off only when *both* sides have them — a symmetric-by-cfg guard.
* `nan_to_zero` is skipped — it's stateless ([02.8 §3](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)), no weights to copy.

And `fit_two_stage` orchestrates it — Stage 1 to its own checkpoint subdir, handoff, then Stage 2:

In [ ]:
life = Path("../../src/neural_data_decoding/training/lifecycle.py").read_text().splitlines()
i = next(j for j, line in enumerate(life) if line.startswith("def fit_two_stage"))
# print the docstring's opening lines (the two-stage summary)
k = next(j for j in range(i, len(life)) if "Stage 1 unsupervised pre-training" in life[j])
for j in range(k, min(k + 8, len(life))):
    print(f"{j + 1:4} | {life[j]}")

Note the **per-stage** checkpointing: Stage 1 writes to `<checkpoint_dir>/stage1_autoencoder/`, Stage 2 to `<checkpoint_dir>/` directly ([05.5](05.5_checkpoint_resume_state_machine.ipynb)'s state machine, twice). So a run interrupted mid-Stage-2 resumes Stage 2 without redoing Stage 1 — the completed Stage 1 checkpoint short-circuits its re-run. Two independent resume points, one lifecycle.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the handoff transplants everything but the classifier

After `copy_autoencoder_weights`, confirm Stage 2's encoder/bottleneck/decoder match Stage 1's, but the classifier is untouched (it has no Stage 1 counterpart).

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
s1 = build_variational_autoencoder(cfg)
s2 = build_variational_composite(cfg)
with torch.no_grad():
    for p in s1.parameters(): p.add_(0.5)      # make Stage 1 distinctive
copy_autoencoder_weights(s1, s2)
for name in ("encoder", "bottleneck", "decoder"):
    same = torch.equal(next(getattr(s1, name).parameters()), next(getattr(s2, name).parameters()))
    print(f"  {name:11}: Stage2 matches Stage1? {same}")
print("  classifier : only exists in Stage 2 (no Stage 1 counterpart to copy)")

### Exercise 4.2 — skip Stage 1

Explain (in a comment) what `num_epochs_autoencoder = 0` does to the lifecycle, and why you'd choose it. (No code — reason it out from §1.)

In [ ]:
# Reveal:
# num_epochs_autoencoder = 0 SKIPS Stage 1 entirely — no unsupervised pre-training.
# Stage 2 then trains the full composite from RANDOM encoder weights (04.8's He init).
# You'd choose it when: labels are plentiful (pre-training buys little), you want a
# faster/simpler run, or you're establishing a from-scratch baseline to measure what
# pre-training actually adds. The C_optimal_synthetic config uses 0; C_two_stage uses >0.
print("reasoned above — no computation needed")

## Section 5 — Common errors

### `RuntimeError` during the handoff

Stage 1 and Stage 2 built from different cfgs — mismatched encoder/bottleneck/decoder shapes ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)). Both must come from the same cfg so their submodule state_dicts align.

### Stage 2 doesn't benefit from Stage 1

The handoff didn't run, or ran into a fresh Stage 2 that was then re-initialized. Confirm encoders match right after `copy_autoencoder_weights` (§2.3).

### Editing Stage 2 changes Stage 1 (or vice versa)

That would mean weights are shared, not copied — they're not (§2.4). If you see this, you aliased the modules somewhere instead of using the state_dict copy.

### Resume redoes Stage 1 every time

Stage 1's checkpoint subdir isn't being found — check `stage1_subdir` and that Stage 1 actually completed (wrote its Current). A completed Stage 1 should short-circuit on re-run.

### Skipping Stage 1 but expecting pre-trained weights

`num_epochs_autoencoder = 0` means NO pre-training — Stage 2 starts random (Exercise 4.2). If you want the warm start, set it positive.

## Section 6 — Further reading

- [Transfer learning tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html) — the pre-train-then-fine-tune pattern in general.
- [`copy_autoencoder_weights`](../../src/neural_data_decoding/models/composite.py) — the handoff.
- [`fit_two_stage`](../../src/neural_data_decoding/training/lifecycle.py) — the two-stage orchestrator with per-stage resume.

Next up: **[05.7 batch-norm state synchronization](05.7_batch_norm_state_synchronization.ipynb)** — the last notebook of Module 05, on the running statistics that update *outside* the gradient path.